# MTL for DNSSEC: Deduplication Optimization Enabled by SigTag

<div style="border-left: 5px solid #FFC107; background-color: #FFFDF5; padding: 12px; border-radius: 4px; color: #000;">
    <strong>⚠️ Please note:</strong> This notebook builds on <code>1-build_a_mtl.ipynb</code>. Some files used by this notebook are generated by <code>1-build_a_mtl.ipynb</code>.
</div>

This notebook provides a demonstration of the deduplication optimization enabled by SigTag EDNS(0) option with MTL for DNSSEC. 

Demo Scenario:
  * A client queries the A record for a domain with the SigTag EDNS(0) option enabled. Because the client does not have any knowledge of ladders, it includes an empty SigTag EDNS(0) option in its query. 
  * The server determines that the domain the client queries does not exist. Therefore, the server returns three NSEC3 records. Since the authentication paths of these NSEC3 records are verified using the same signed ladder, the server performs deduplication optimization by including the common signed ladder within a full signature only once. The remaining are condensed signatures. The server's response also includes an empty SigTag EDNS(0) option to signal its support for SigTag. 

## 📋 Demo Outline

<div class="alert alert-block alert-info" style="border-left: 20px solid #a8a7a8; border-right: solid #a8a7a8; border-top: solid #a8a7a8; border-bottom: solid #a8a7a8; background-color: #f1f0f2; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #757475;">Background and Demo Configurations</h3>
<ul>
    <li>Deduplication Optimization</li>
    <li>Importing Libraries</li>
    <li>Demo Configuration</li>
        <ul>
            <li>Hashing Configurations</li>
            <li>Node Set</li>
            <li>Example SOA Record</li>
            <li>Example List of Hashes of the Original Ownernames in the Zone</li>
            <li>Example NSEC3 Records</li>
            <li>Signed Ladder</li>
        </ul>
</ul>
</div>

<div class="alert alert-block alert-danger" style="border-left: 20px solid #c7745d; border-right: solid #c7745d; border-top: solid #c7745d; border-bottom: solid #c7745d; background-color: #f7f1ef; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #c97058;">Handling SigTag</h3>
<ul>
    <li><strong>Section 1</strong> Client - Requesting the A Record of Domain <code>a.c.x.w.example</code> with SigTag Enabled</li>
    <li><strong>Section 2</strong> Server - Handling of the Client Request</li>
</ul>
</div>

<div class="alert alert-block alert-success" style="border-left: 20px solid #5c86a8; border-right: solid #5c86a8; border-top: solid #5c86a8; border-bottom: solid #5c86a8; background-color: #f5f9fc; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #4e7da3;">DNSSEC Signed Data and Signed Ladder</h3>
<ul>
    <li><strong>Section 3</strong> Server - Generating the DNSSEC Signed Data</li>
        <ul>
            <li><strong>Section 3.1</strong> Generating DNSSEC Signed Data for the SOA Record</li>
            <li><strong>Section 3.2</strong> Generating DNSSEC Signed Data for the NSEC3 Records</li>
        </ul> 
    <li><strong>Section 4</strong> Server - Obtaining the Signed Ladder</li>
</ul>
</div>


<div class="alert alert-block alert-danger" style="border-left: 20px solid #739e51; border-right: solid #739e51; border-top: solid #739e51; border-bottom: solid #739e51; background-color: #f6fcf2; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #5d8043;">Constructing Authentication Paths and One Full Signature</h3>
<ul>
    <li><strong>Section 5</strong> Server - Constructing the Authentication Path for the SOA Record</li>
        <ul>
            <li><strong>Section 5.1</strong> Obtaining the Leaf Node Hash of the SOA Record</li>
            <li><strong>Section 5.2</strong> Obtaining the Target Rung for the SOA Record</li>
            <li><strong>Section 5.3</strong> Finding the Siblings for the SOA Record</li>
            <li><strong>Section 5.4</strong> Building the Authentication Path for the SOA Record</li>
        </ul> 
    <li><strong>Section 6</strong> Server - Constructing a Full Signature for the SOA Record</li>
    <li><strong>Section 7</strong> Server - Constructing Authentication Paths for the NSEC3 Records</li>
        <ul>
            <li><strong>Section 7.1</strong> Obtaining Leaf Node Hashes of the NSEC3 Records</li>
            <li><strong>Section 7.2</strong> Obtaining Target Rungs for the NSEC3 Records</li>
            <li><strong>Section 7.3</strong> Finding the Siblings for the NSEC3 Records</li>
            <li><strong>Section 7.4</strong> Building Authentication Paths for the NSEC3 Records</li>
        </ul> 
</ul>
</div>


<div class="alert alert-block alert-warning" style="border-left: 20px solid #87779c; border-right: solid #87779c; border-top: solid #87779c; border-bottom: solid #87779c; background-color: #f9f7fc; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #6f5f82;">Constructing a Response</h3>
<ul>
    <li><strong>Section 8</strong> Server - Constructing a Response</li>
</ul>
</div>

<div class="alert alert-block alert-danger" style="border-left: 20px solid #a8a7a8; border-right: solid #a8a7a8; border-top: solid #a8a7a8; border-bottom: solid #a8a7a8; background-color: #f1f0f2; padding: 15px; margin-bottom: 10px; color: #000;">
<h3 style="margin-top: 0; color: #757475;">Demo Summary</h3>
<ul>
    <li>End of Demo</li>
</ul>
</div>

## 📖 Deduplication Optimization 
> 
> The deduplication optimization will be performed by the server when:
>   * The client supports SigTag, 
>   * Multiple RRSIG records in the response have condensed signatures that can be verified using the same signed ladder, and
>   * The signed ladder does not match any SigTag advertised by the client.
> 
> Specifically, the server will deduplicate its response by including each common signed ladder within a full signature only once. When deduplication is performed, the full signature is present in the first RRSIG. Other RRSIG records within the response will be condensed signatures.

In this demo, we will showcase an NXDOMAIN response with NSEC3 that returns 3 NSEC3 records, as as detailed in https://www.rfc-editor.org/info/rfc5155/, to demostrate how the deduplication optimization works. 

## 📦 Importing Libraries to Run This Demo: 

In [1]:
from IPython.display import display
import json
import struct
import binascii
import hashlib

Helper functions are imported right before execution. 

## ⚙️ Demo Configurations

### (1) Hashing Configurations
The hash algorithm and security parameters must match the ladder configuration.

<div style="border-left: 5px solid #FFC107; background-color: #FFFDF5; padding: 12px; border-radius: 4px; color: #000;">
    <strong>⚠️ Please note:</strong> Field sizes in following sections depend on the signature algorithms and may vary if other algorithms are used, which requires corresponding modifications to the underlying functions. 
</div>

In [2]:
# Hash algorithm for generating node hashes: this demo supports "shake".
hash_alg = "shake"

# Security parameter n: this demo supports n = 16, 24, or 32.
n = 16

### (2) Node Set
In this demo, we use the node set used in `1-build_a_mtl.ipynb`. This node set contains 10 leaf nodes and is located at `data/nodeset.json`. 

In [3]:
nodeset_filepath = "data/nodeset.json"

with open(nodeset_filepath, "r") as f:
    nodeset_data = json.load(f)

Specifically, the leaf node with index 0 corresponds to the Start-of-Authority (SOA) record and nodes with indices 5, 6, and 7 correspond to the NSEC3 records. 

## (3) Example SOA Record

In [4]:
from helper.utils import encode_domain

mname = encode_domain("ns1.example.")                # Master name server
rname = encode_domain("admin.example.")              # Responsible mailbox
soa_timers = struct.pack(
    "!IIIII",
    2026100201,                                      # Serial number
    86400,                                           # Refresh interval
    7200,                                            # Retry interval
    3600000,                                         # Expire limit
    3600                                             # Minimum TTL
)
soa_rdata = mname + rname + soa_timers

soa_rrsig_meta = {
    "type_covered": 6,                               # DNS record type SOA
    "algorithm": 131,                                # Assumed alogirithm ID
    "labels": 1,                                     # "example. " has one label
    "original_ttl": 86400,                           # The TTL the signature is based on
    "sig_expiration": 20261102112500,                # Signature expiration timestamp
    "sig_inception": 20261002112500,                 # Signature inception timestamp
    "key_tag": 46491,                                # ID of the DNSKEY used for signing
    "signer_name": "example."                        # Signer's name
}

soa_rrset = [
    {
        "name": "example.", 
        "type": 6,                                   # DNS record type SOA
        "class": 1,                                  # IN class
        "rdata": soa_rdata                           # Formatted binary payload
    },
]

## (4) Example List of Hashes of the Original Ownernames in the Zone
```
H(example)       = 0p9mhaveqvm6t7vbl5lop2u3t2rp3tom
H(a.example)     = 35mthgpgcu1qg68fab165klnsnk3dpvl
H(ns1.example)   = 2t7b4g4vsa5smi47k61mv5bv1a22bojr
H(ns2.example)   = q04jkcevqvmu85r014c7dkba38o0ji5r
H(*.w.example)   = r53bq7cc2uvmubfu5ocmm6pers9tk9en
H(x.w.example)   = b4um86eghhds6nea196smvmlo4ors995
H(y.w.example)   = ji6neoaepv8b5o6k4ev33abha8ht9fgc
H(x.y.w.example) = 2vptu5timamqttgl4luu9kg21e0aor3s
H(xx.example)    = t644ebqk9bibcna874givr6joj62mlhv
H(2t7b4g4vsa5smi47k61mv5bv1a22bojr.example)
                  = kohar7mbb8dc2ce8a9qvl8hon4k53uhi
```

## (5) Example NSEC3 Records

Three NSEC3 records will be used in this demo. 

In [5]:
# Shared properties across all three NSEC3 records

hash_alg = 1        # SHA-1
flags = 1           # Opt-Out
iterations = 0
salt = binascii.unhexlify("aabbccdd")
header_pack = struct.pack("!BBHB", hash_alg, flags, iterations, len(salt)) + salt

In [6]:
from helper.utils import decode_b32_nsec3

# ---------------------------------------------------------------------------
# NSEC3 RECORD 1: Covers "next closer" name (c.x.w.example in this demo)
# ---------------------------------------------------------------------------

# Assume next hashed owner: 2t7b4g4vsa5smi47k61mv5bv1a22bojr
next_owner_1 = decode_b32_nsec3("2t7b4g4vsa5smi47k61mv5bv1a22bojr")
# Pre-calculated bitmap - window 0, len 7
bitmap_1 = b"\x00\x07\x22\x01\x00\x00\x00\x02\x90"

rdata_1 = header_pack + bytes([len(next_owner_1)]) + next_owner_1 + bitmap_1

nsec3_1_meta = {
    "type_covered": 50,
    "algorithm": 131,
    "labels": 1, 
    "original_ttl": 3600,
    "sig_expiration": 20150420235959,
    "sig_inception": 20051021000000,
    "key_tag": 46491,
    "signer_name": "example."
}

# H(c.x.w.example) = 0va5bpr2ou0vk0lbqeeljri88laipsfh
# H(example)       = 0p9mhaveqvm6t7vbl5lop2u3t2rp3tom
nsec3_1_rrset = [
    {
        "name": "0p9mhaveqvm6t7vbl5lop2u3t2rp3tom.example.",
        "type": 50,
        "class": 1,
        "rdata": rdata_1
    }
]


# ---------------------------------------------------------------------------
# NSEC3 RECORD 2: Matches the closest encloser (x.w.example)
# ---------------------------------------------------------------------------

# Assume next hashed owner: gjeqe526plbf1g8mklp59enfd789njgi
next_owner_2 = decode_b32_nsec3("gjeqe526plbf1g8mklp59enfd789njgi")
# Pre-calculated bitmap - window 0, len 6
bitmap_2 = b"\x00\x06\x00\x01\x00\x00\x00\x02"

rdata_2 = header_pack + bytes([len(next_owner_2)]) + next_owner_2 + bitmap_2

nsec3_2_meta = {
    "type_covered": 50,
    "algorithm": 131,
    "labels": 1,
    "original_ttl": 3600,
    "sig_expiration": 20150420235959,
    "sig_inception": 20051021000000,
    "key_tag": 46491,
    "signer_name": "example."
}

# H(x.w.example) = b4um86eghhds6nea196smvmlo4ors995
nsec3_2_rrset = [
    {
        "name": "b4um86eghhds6nea196smvmlo4ors995.example.",
        "type": 50,
        "class": 1,
        "rdata": rdata_2
    }
]


# ---------------------------------------------------------------------------
# NSEC3 RECORD 3: Covers wildcard at the closest encloser (*.x.w.example)
# ---------------------------------------------------------------------------

# Assume next hashed owner: b4um86eghhds6nea196smvmlo4ors995
next_owner_3 = decode_b32_nsec3("b4um86eghhds6nea196smvmlo4ors995")
# Pre-calculated bitmap - window 0, len 6
bitmap_3 = b"\x00\x06\x20\x00\x00\x00\x00\x12"

rdata_3 = header_pack + bytes([len(next_owner_3)]) + next_owner_3 + bitmap_3

nsec3_3_meta = {
    "type_covered": 50,
    "algorithm": 131,
    "labels": 1,
    "original_ttl": 3600,
    "sig_expiration": 20150420235959,
    "sig_inception": 20051021000000,
    "key_tag": 46491,
    "signer_name": "example."
}

# H(*.x.w.example) = 92pqneegtaue7pjatc3l3qnk738c6v5m
# H(a.example)     = 35mthgpgcu1qg68fab165klnsnk3dpvl
nsec3_3_rrset = [
    {
        "name": "35mthgpgcu1qg68fab165klnsnk3dpvl.example.",
        "type": 50,
        "class": 1,
        "rdata": rdata_3
    }
]

## (6) Signed Ladder

In [7]:
with open("data/ladder_signed.txt", "r") as f:
    ladder_readin = f.read()
signed_ladder = bytes.fromhex(ladder_readin)

## 1. Client - Requesting the A Record of Domain `a.c.x.w.example` with SigTag Enabled

The client queries for the A record of the domain `a.c.x.w.example` with the SigTag EDNS(0) option. Because the client does not have any known ladders, it includes an **empty** SigTag EDNS(0) option in its DNS query, as shown below:
```
   0                       8                      16
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
   |               OPTION-CODE = TBD               |
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
   |             OPTION-LENGTH = 0x0000            |
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
```

## 2. Server - Handling of the Client Request

Upon receiving the client's DNS query, the server determines that the domain the client requests **does not exist**. Therefore, the server prepares a response that includes three NSEC3 records for: 
  * `c.x.w.example.org`: the "next closer" name
  * `x.w.example.org`: the closest encloser
  * `*.x.w.example.org`: wildcard at the closest encloser

The server also includes a SOA record in its response. Since the SOA and NSEC3 records are leaf nodes of the same MTL, they are covered by the same signature (i.e., signed ladder). Because the client supports SigTag, the server then performs the deduplication optimization by **including the signed ladder within a full signature only once** in its response. 

In [8]:
from helper.mtl_parser import Ladder

with open("data/ladder_signed.txt", "r") as f:
    ladder_servreadin = f.read()
ladder_servreadin_bytes = bytes.fromhex(ladder_servreadin)
ladder_signed = Ladder(fullsig=ladder_servreadin_bytes, n=n)

The server also stores the SigTag value of this signed ladder to validate it against the client-provided SigTag, if any: 

In [9]:
from helper.utils import print_data

sigtag_server = ladder_signed.sid + ladder_signed.rindex
print_data("SigTag Value of Server Signed Ladder", sigtag_server.hex(), len(sigtag_server), n=n, print_type="sigtag")

┌─────────────────────────────────────┐
│  ■ SID   ■ Ladder Right Index       │
└─────────────────────────────────────┘

╭────────────────── SigTag Value of Server Signed Ladder ───────────────────╮
│                                                                           │
│  8F5CE818 A1DF091D FB8288CC AD6FF450 99A48C46 4F540CAB 05114981 736BB9D0  │
│  00000000 00000009                                                        │
│                                                                           │
│                                                                           │
│  Length of SigTag Value of Server Signed Ladder: 40 bytes                 │
│                                                                           │
╰───────────────────────────────────────────────────────────────────────────╯

## 3. Server - Generating the DNSSEC Signed Data
The server first generates the DNSSEC signed data. 

### 3.1 Generating DNSSEC Signed Data for the SOA Record

In [10]:
from helper.utils import create_dnssec_signed_data

message_soa = create_dnssec_signed_data(soa_rrsig_meta, soa_rrset)
print_data("DNSSEC Signed Data for the SOA Record", message_soa.hex(), len(message_soa))

╭────────────────── DNSSEC Signed Data for the SOA Record ──────────────────╮
│                                                                           │
│  00068301 00015180 6AE8730C 6ABF948C B59B0765 78616D70 6C650007 6578616D  │
│  706C6500 00060001 00015180 0030036E 73310765 78616D70 6C650005 61646D69  │
│  6E076578 616D706C 650078C3 D5E90001 51800000 1C200036 EE800000 0E10      │
│                                                                           │
│                                                                           │
│  Length of DNSSEC Signed Data for the SOA Record: 94 bytes                │
│                                                                           │
╰───────────────────────────────────────────────────────────────────────────╯

### 3.2 Generating DNSSEC Signed Data for the NSEC3 Records

In [11]:
nsec3_records = [
    ("nsec3_1", "NSEC3 Record 1 - Next Closer", create_dnssec_signed_data(nsec3_1_meta, nsec3_1_rrset)),
    ("nsec3_2", "NSEC3 Record 2 - Closest Encloser", create_dnssec_signed_data(nsec3_2_meta, nsec3_2_rrset)),
    ("nsec3_3", "NSEC3 Record 3 - Wildcard Cover", create_dnssec_signed_data(nsec3_3_meta, nsec3_3_rrset))
]

for _, description, msg in nsec3_records:
    print_data(f"DNSSEC Signed Data for {description}", msg.hex(), len(msg))

╭─────────── DNSSEC Signed Data for NSEC3 Record 1 - Next Closer ────────────╮
│                                                                            │
│  00328301 00000E10 553592FF 43582F80 B59B0765 78616D70 6C650020 3070396D   │
│  68617665 71766D36 74377662 6C356C6F 70327533 74327270 33746F6D 07657861   │
│  6D706C65 00003200 0100000E 10002701 01000004 AABBCCDD 14174EB2 409FE28B   │
│  CB4887A1 836F957F 0A8425E2 7B000722 01000000 0290                         │
│                                                                            │
│                                                                            │
│  Length of DNSSEC Signed Data for NSEC3 Record 1 - Next Closer: 118 bytes  │
│                                                                            │
╰────────────────────────────────────────────────────────────────────────────╯

╭─────────── DNSSEC Signed Data for NSEC3 Record 2 - Closest Encloser ────────────╮
│                                                                                 │
│  00328301 00000E10 553592FF 43582F80 B59B0765 78616D70 6C650020 6234756D        │
│  38366567 68686473 366E6561 31393673 6D766D6C 6F346F72 73393935 07657861        │
│  6D706C65 00003200 0100000E 10002601 01000004 AABBCCDD 1484DDA7 1446CD56        │
│  F0C116A5 7254BAEF 69D09BCE 12000600 01000000 02                                │
│                                                                                 │
│                                                                                 │
│  Length of DNSSEC Signed Data for NSEC3 Record 2 - Closest Encloser: 117 bytes  │
│                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────╯

╭─────────── DNSSEC Signed Data for NSEC3 Record 3 - Wildcard Cover ────────────╮
│                                                                               │
│  00328301 00000E10 553592FF 43582F80 B59B0765 78616D70 6C650020 33356D74      │
│  68677067 63753171 67363866 61623136 356B6C6E 736E6B33 6470766C 07657861      │
│  6D706C65 00003200 0100000E 10002601 01000004 AABBCCDD 14593D64 19D08C5B      │
│  C35DCA0A 4DCB7ED5 C131BE25 25000620 00000000 12                              │
│                                                                               │
│                                                                               │
│  Length of DNSSEC Signed Data for NSEC3 Record 3 - Wildcard Cover: 117 bytes  │
│                                                                               │
╰───────────────────────────────────────────────────────────────────────────────╯

## 4. Server - Obtaining the Signed Ladder

As mentioned above, the SOA and NSEC3 records are leaf nodes of the same MTL. Therefore, they share the same ladder. 

In [12]:
from helper.mtl_parser import Ladder

display(ladder_signed.ladder_print())

Ladder Field,Field Length (bytes),Field Data


## 5. Server - Constructing the Authentication Path for the SOA Record

An authentication path consists of: 

```
                     1 1 1 1 1 1
 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5
+-------------------------------+
|             Flags             |
+-------------------------------+
|                               |
//             SID             //
|                               |
+-------------------------------+
|                               |
//          Randomizer         //
|                               |
+-------------------------------+
|                               |
|           Leaf Index          |
|                               |
|                               |
+-------------------------------+
|                               |
|     Target Rung Left Index    |
|                               |
|                               |
+-------------------------------+
|                               |
|    Target Rung Right Index    |
|                               |
|                               |
+-------------------------------+
|       Sibling Hash Count      |
+-------------------------------+
|                               |
//      Sibling Hash Values    //
|                               |
+-------------------------------+
```

In this demo, the SOA record is the leaf node with index 0. The randomizer is associated to this specific leaf node and impacts the resulting leaf hash value. 

In [13]:
soa_leaf_index = 0

In [14]:
from helper.ladder_updater import LeafNode

soa_leaf_node = LeafNode(lindex=soa_leaf_index, rindex=soa_leaf_index)
soa_leaf_node.randomizer = bytes.fromhex(nodeset_data["nodes"]["level_0"][soa_leaf_index]["randomizer"])

### 5.1 Obtaining the Leaf Node Hash of the SOA Record

In [15]:
soa_leaf_node.node_hash = bytes.fromhex(nodeset_data["nodes"]["level_0"][soa_leaf_node.lindex]["hash"])
display(soa_leaf_node.leaf_print())

"Leaf Node (0, 0)",Value


### 5.2 Obtaining the Target Rung for the SOA Record

In [16]:
from helper.ladder_updater import find_target_rung

soa_target_rung = find_target_rung(nodeset=nodeset_data, leaf_index=soa_leaf_node.lindex)

soa_leaf_node.rung_lindex_int = soa_target_rung["left_index"]
soa_leaf_node.rung_rindex_int = soa_target_rung["right_index"]
soa_leaf_node.rung_lindex = soa_leaf_node.rung_lindex_int.to_bytes(8, byteorder="big")
soa_leaf_node.rung_rindex = soa_leaf_node.rung_rindex_int.to_bytes(8, byteorder="big")
soa_leaf_node.rung_hash = bytes.fromhex(soa_target_rung["hash"])
display(soa_leaf_node.targetrung_print())

"Leaf Node (0, 0) Target Rung",Value


### 5.3 Finding the Siblings for the SOA Record

In [17]:
from helper.ladder_updater import get_sibling_nodes

soa_sibling_list = get_sibling_nodes(nodeset=nodeset_data, leaf_index=soa_leaf_node.lindex)

soa_leaf_node.siblings = []
soa_leaf_node.siblings_withindex = []

if not soa_sibling_list:
    print(f"\033[1;31mThe Leaf Node ({leaf_node.lindex}, {leaf_node.rindex}) does not have siblings\033[0m.")
else: 
    for idx, sib in enumerate(soa_sibling_list):
        soa_leaf_node.siblings.append(bytes.fromhex(sib["hash"]))
        soa_leaf_node.siblings_withindex.append({
            "left_index": sib["left_index"],
            "right_index": sib["right_index"],
            "hash": bytes.fromhex(sib["hash"])
        })
    display(soa_leaf_node.leafsiblings_print())

"Leaf Node (0, 0) Sibling",Value


### 5.4 Building the Authentication Path for the SOA Record

The server next builds the authentication path for this leaf node as follows:

In [18]:
# Flag has to be 0
soa_authpath_bytes = int("0").to_bytes(2, byteorder="big") + \
                     ladder_signed.sid + \
                     soa_leaf_node.randomizer + \
                     soa_leaf_node.lindex.to_bytes(8, byteorder="big") + \
                     soa_leaf_node.rung_lindex + \
                     soa_leaf_node.rung_rindex + \
                     len(soa_leaf_node.siblings).to_bytes(2, byteorder="big")

if soa_leaf_node.siblings:
    for each_sib_hash in soa_leaf_node.siblings:
        soa_authpath_bytes += each_sib_hash

soa_leaf_node.authpath = soa_authpath_bytes
display(soa_leaf_node.authpath_print())

"Leaf Node (0, 0) Authentication Path",Value


## 6. Server - Constructing a Full Signature for the SOA Record

The full signature should start with the `MTL-Type` of 1, as specified in draft-kaizer-dnsop-ml-dsa-mtl-dnssec-03, followed by components listed below:
```
                     1 1 1 1 1 1
 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5
+-------------------------------+
|                               |
//     Condensed Signature     //
|     (Authentication Path)     |
+-------------------------------+
|                               |
//        Signed Ladder        //
|                               |
+-------------------------------+
```

In [19]:
full_sig = int(1).to_bytes(1, byteorder="big") + soa_leaf_node.authpath + ladder_servreadin_bytes
print_data("Full Signature", full_sig.hex(), len(full_sig), n=n, print_type="fullsig")

┌──────────────────────────────────────────────────────────────────────────────────────┐
│  ■ MTL-Type        ■ Flags               ■ SID                  ■ Randomizer         │
│  ■ Leaf Index      ■ Target Rung Index   ■ Sibling Hash Count   ■ Sibling Hash       │
│  ■ Flags           ■ SID                 ■ Rung Count           ■ Rung [1] Index     │
│  ■ Rung [1] Hash   ■ Rung [2] Index      ■ Rung [2] Hash        ■ Signature Length   │
│  ■ Signature                                                                         │
└──────────────────────────────────────────────────────────────────────────────────────┘

╭───────────────────────────── Full Signature ──────────────────────────────╮
│                                                                           │
│  0100008F 5CE818A1 DF091DFB 8288CCAD 6FF45099 A48C464F 540CAB05 11498173  │
│  6BB9D0F4 E32A88B1 9CD75F63 1A0E4B88 229C1100 00000000 00000000 00000000  │
│  00000000 00000000 00000700 036EAD94 B673EA28 7A47B269 C1700479 352B7BCB  │
│  12A626E0 74F2676E CD0FF6A7 CD33D3A6 54E1A31A 4D91EF3C 28478A6C C800008F  │
│  5CE818A1 DF091DFB 8288CCAD 6FF45099 A48C464F 540CAB05 11498173 6BB9D000  │
│  02000000 00000000 00000000 00000000 078080DF D002E93A C7994F0C B974FFE7  │
│  04000000 00000000 08000000 00000000 09B015B4 429BEC7E 08EE28B2 8A79C453  │
│  A5000009 74AEE67B 06EE4BC5 87BE982E 29AF93DC 156F24E4 936833C1 7703C53F  │
│  8BE57A16 0522CEF5 B9064CBC 1FF46F1E 86A0B08A 3AFC4FC5 4D68A8D5 95ECD4DB  │
│  052B866A 989B422D 4848271A 63CC577B 2D9822FA 0426E738 17E1C43B 7EA7634C  │
│  97C745E4 EB348B3B D859B52D 5D46A213 581F515B FF9AE0EC 9630A52C 6CFE07FB  │
│  C1106573 B6B4B484 5529878A 6EDB5F9E 1E61541A 5A6C8EB3 30072801 15BC512B  │
│  B160E5D8 49C84D39 96228EFA 0200FE71 9B45D0AA CF996B24 D4384BA3 CC4FC290  │
│  27C38902 6170097E F3702B14 55C17AE3 15EC5CB2 20B87AB0 6E98AD98 DC8CFD9D  │
│  BF70BBE7 CAFFF2F0 55DA5E49 511A918F 631CA281 E0ABBEA1 4EFD1141 D74799C5  │
│  7936C3AB 7F86424A E4D2F138 DB338C0D C4D6C6A0 E477D863 220AB801 AC486344  │
│  02292768 DFDE0736 C0B11751 1B42DA18 0B873C59 AB25AC2E F9B6FDCF FE171274  │
│  45B932C1 EFCA7D45 720593C4 3FF75FE2 5A752271 C7AB9D5D FDAE0724 2C485FA5  │
│  6F6E31F9 FB6B6C0E EBDF4A4D 05E9D0E8 05215907 3031AE89 8EEE5BC4 D37EE1BC  │
│  DCFA09E6 7896539A 86D4F47A 3BACEC5D F4635564 233AC2A8 7FD6C8B8 624BE3A2  │
│  368D4E71 76650B9E EEB30282 DFA7AEE3 2F0D57E6 77E7E199 3BE5DD03 C16EA69B  │
│  [1;38;2;180;181;181;48

## 7. Server - Constructing Authentication Paths for the NSEC3 Records

In this demo, the NSEC3 records are leaf nodes with indices 5, 6, and 7. The randomizer is associated to each specific leaf node and impacts the resulting leaf hash values. 

In [20]:
nsec3_1_leaf_index = 5
nsec3_2_leaf_index = 6
nsec3_3_leaf_index = 7

In [21]:
from helper.ladder_updater import LeafNode

nsec3_1_leaf_node = LeafNode(lindex=nsec3_1_leaf_index, rindex=nsec3_1_leaf_index)
nsec3_1_leaf_node.randomizer = bytes.fromhex(nodeset_data["nodes"]["level_0"][nsec3_1_leaf_index]["randomizer"])

nsec3_2_leaf_node = LeafNode(lindex=nsec3_2_leaf_index, rindex=nsec3_2_leaf_index)
nsec3_2_leaf_node.randomizer = bytes.fromhex(nodeset_data["nodes"]["level_0"][nsec3_2_leaf_index]["randomizer"])

nsec3_3_leaf_node = LeafNode(lindex=nsec3_3_leaf_index, rindex=nsec3_3_leaf_index)
nsec3_3_leaf_node.randomizer = bytes.fromhex(nodeset_data["nodes"]["level_0"][nsec3_3_leaf_index]["randomizer"])

### 7.1 Obtaining Leaf Node Hashes of the NSEC3 Records

In [22]:
nsec3_1_leaf_node.node_hash = bytes.fromhex(nodeset_data["nodes"]["level_0"][nsec3_1_leaf_node.lindex]["hash"])
display(nsec3_1_leaf_node.leaf_print())

nsec3_2_leaf_node.node_hash = bytes.fromhex(nodeset_data["nodes"]["level_0"][nsec3_2_leaf_node.lindex]["hash"])
display(nsec3_2_leaf_node.leaf_print())

nsec3_3_leaf_node.node_hash = bytes.fromhex(nodeset_data["nodes"]["level_0"][nsec3_3_leaf_node.lindex]["hash"])
display(nsec3_3_leaf_node.leaf_print())

"Leaf Node (5, 5)",Value


"Leaf Node (6, 6)",Value


"Leaf Node (7, 7)",Value


### 7.2 Obtaining Target Rungs for the NSEC3 Records

In [23]:
from helper.ladder_updater import find_target_rung

nsec3_1_target_rung = find_target_rung(nodeset=nodeset_data, leaf_index=nsec3_1_leaf_node.lindex)
nsec3_1_leaf_node.rung_lindex_int = nsec3_1_target_rung["left_index"]
nsec3_1_leaf_node.rung_rindex_int = nsec3_1_target_rung["right_index"]
nsec3_1_leaf_node.rung_lindex = nsec3_1_leaf_node.rung_lindex_int.to_bytes(8, byteorder="big")
nsec3_1_leaf_node.rung_rindex = nsec3_1_leaf_node.rung_rindex_int.to_bytes(8, byteorder="big")
nsec3_1_leaf_node.rung_hash = bytes.fromhex(nsec3_1_target_rung["hash"])
display(nsec3_1_leaf_node.targetrung_print())

nsec3_2_target_rung = find_target_rung(nodeset=nodeset_data, leaf_index=nsec3_2_leaf_node.lindex)
nsec3_2_leaf_node.rung_lindex_int = nsec3_2_target_rung["left_index"]
nsec3_2_leaf_node.rung_rindex_int = nsec3_2_target_rung["right_index"]
nsec3_2_leaf_node.rung_lindex = nsec3_2_leaf_node.rung_lindex_int.to_bytes(8, byteorder="big")
nsec3_2_leaf_node.rung_rindex = nsec3_2_leaf_node.rung_rindex_int.to_bytes(8, byteorder="big")
nsec3_2_leaf_node.rung_hash = bytes.fromhex(nsec3_2_target_rung["hash"])
display(nsec3_2_leaf_node.targetrung_print())

nsec3_3_target_rung = find_target_rung(nodeset=nodeset_data, leaf_index=nsec3_3_leaf_node.lindex)
nsec3_3_leaf_node.rung_lindex_int = nsec3_3_target_rung["left_index"]
nsec3_3_leaf_node.rung_rindex_int = nsec3_3_target_rung["right_index"]
nsec3_3_leaf_node.rung_lindex = nsec3_3_leaf_node.rung_lindex_int.to_bytes(8, byteorder="big")
nsec3_3_leaf_node.rung_rindex = nsec3_3_leaf_node.rung_rindex_int.to_bytes(8, byteorder="big")
nsec3_3_leaf_node.rung_hash = bytes.fromhex(nsec3_3_target_rung["hash"])
display(nsec3_3_leaf_node.targetrung_print())

"Leaf Node (5, 5) Target Rung",Value


"Leaf Node (6, 6) Target Rung",Value


"Leaf Node (7, 7) Target Rung",Value


### 7.3 Finding the Siblings for the NSEC3 Records

In [24]:
from helper.ladder_updater import get_sibling_nodes

nsec3_1_sibling_list = get_sibling_nodes(nodeset=nodeset_data, leaf_index=nsec3_1_leaf_node.lindex)

nsec3_1_leaf_node.siblings = []
nsec3_1_leaf_node.siblings_withindex = []

if not nsec3_1_sibling_list:
    print(f"The Leaf Node ({nsec3_1_leaf_node.lindex}, {nsec3_1_leaf_node.rindex}) does not have siblings.")
else: 
    for idx, sib in enumerate(nsec3_1_sibling_list):
        nsec3_1_leaf_node.siblings.append(bytes.fromhex(sib["hash"]))
        nsec3_1_leaf_node.siblings_withindex.append({
            "left_index": sib["left_index"],
            "right_index": sib["right_index"],
            "hash": bytes.fromhex(sib["hash"])
        })
    display(nsec3_1_leaf_node.leafsiblings_print())

"Leaf Node (5, 5) Sibling",Value


In [25]:
from helper.ladder_updater import get_sibling_nodes

nsec3_2_sibling_list = get_sibling_nodes(nodeset=nodeset_data, leaf_index=nsec3_2_leaf_node.lindex)

nsec3_2_leaf_node.siblings = []
nsec3_2_leaf_node.siblings_withindex = []

if not nsec3_2_sibling_list:
    print(f"The Leaf Node ({nsec3_2_leaf_node.lindex}, {nsec3_2_leaf_node.rindex}) does not have siblings.")
else: 
    for idx, sib in enumerate(nsec3_2_sibling_list):
        nsec3_2_leaf_node.siblings.append(bytes.fromhex(sib["hash"]))
        nsec3_2_leaf_node.siblings_withindex.append({
            "left_index": sib["left_index"],
            "right_index": sib["right_index"],
            "hash": bytes.fromhex(sib["hash"])
        })
    display(nsec3_2_leaf_node.leafsiblings_print())

"Leaf Node (6, 6) Sibling",Value


In [26]:
from helper.ladder_updater import get_sibling_nodes

nsec3_3_sibling_list = get_sibling_nodes(nodeset=nodeset_data, leaf_index=nsec3_3_leaf_node.lindex)

nsec3_3_leaf_node.siblings = []
nsec3_3_leaf_node.siblings_withindex = []

if not nsec3_3_sibling_list:
    print(f"The Leaf Node ({nsec3_3_leaf_node.lindex}, {nsec3_3_leaf_node.rindex}) does not have siblings.")
else: 
    for idx, sib in enumerate(nsec3_3_sibling_list):
        nsec3_3_leaf_node.siblings.append(bytes.fromhex(sib["hash"]))
        nsec3_3_leaf_node.siblings_withindex.append({
            "left_index": sib["left_index"],
            "right_index": sib["right_index"],
            "hash": bytes.fromhex(sib["hash"])
        })
    display(nsec3_3_leaf_node.leafsiblings_print())

"Leaf Node (7, 7) Sibling",Value


### 7.4 Building Authentication Paths for the NSEC3 Records

The server next builds the authentication paths for three NSEC3 leaf node as follows:

In [27]:
# Flag has to be 0
nsec3_1_authpath_bytes = int("0").to_bytes(2, byteorder="big") + \
                         ladder_signed.sid + \
                         nsec3_1_leaf_node.randomizer + \
                         nsec3_1_leaf_node.lindex.to_bytes(8, byteorder="big") + \
                         nsec3_1_leaf_node.rung_lindex + \
                         nsec3_1_leaf_node.rung_rindex + \
                         len(nsec3_1_leaf_node.siblings).to_bytes(2, byteorder="big")

if nsec3_1_leaf_node.siblings:
    for each_sib_hash in nsec3_1_leaf_node.siblings:
        nsec3_1_authpath_bytes += each_sib_hash

nsec3_1_leaf_node.authpath = nsec3_1_authpath_bytes
display(nsec3_1_leaf_node.authpath_print())

cond_sig1 = int(2).to_bytes(1, byteorder="big") + nsec3_1_leaf_node.authpath
print_data("Condensed Signature For 'Next Closer' Name", 
           cond_sig1.hex(), len(cond_sig1), n=n, print_type="condsig")

"Leaf Node (5, 5) Authentication Path",Value


┌───────────────────────────────────────────────────────────────────────────────┐
│  ■ MTL-Type     ■ Flags               ■ SID                  ■ Randomizer     │
│  ■ Leaf Index   ■ Target Rung Index   ■ Sibling Hash Count   ■ Sibling Hash   │
└───────────────────────────────────────────────────────────────────────────────┘

╭─────────────── Condensed Signature For 'Next Closer' Name ────────────────╮
│                                                                           │
│  0200008F 5CE818A1 DF091DFB 8288CCAD 6FF45099 A48C464F 540CAB05 11498173  │
│  6BB9D08A 05C3D4F7 E219BA88 301ECBC9 94D50200 00000000 00000500 00000000  │
│  00000000 00000000 00000700 03CEB61A 79CEFAFC 18BE0849 5D1D9F21 2D1432F7  │
│  A9A76009 432B0E20 50727FDC 89AD692E 7D45185C E82BF093 144A22C7 15        │
│                                                                           │
│                                                                           │
│  Length of Condensed Signature For 'Next Closer' Name: 125 bytes          │
│                                                                           │
╰───────────────────────────────────────────────────────────────────────────╯

In [28]:
# Flag has to be 0
nsec3_2_authpath_bytes = int("0").to_bytes(2, byteorder="big") + \
                         ladder_signed.sid + \
                         nsec3_2_leaf_node.randomizer + \
                         nsec3_2_leaf_node.lindex.to_bytes(8, byteorder="big") + \
                         nsec3_2_leaf_node.rung_lindex + \
                         nsec3_2_leaf_node.rung_rindex + \
                         len(nsec3_2_leaf_node.siblings).to_bytes(2, byteorder="big")

if nsec3_2_leaf_node.siblings:
    for each_sib_hash in nsec3_2_leaf_node.siblings:
        nsec3_2_authpath_bytes += each_sib_hash

nsec3_2_leaf_node.authpath = nsec3_2_authpath_bytes
display(nsec3_2_leaf_node.authpath_print())

cond_sig2 = int(2).to_bytes(1, byteorder="big") + nsec3_2_leaf_node.authpath
print_data("Condensed Signature For 'the Closest Encloser'", 
           cond_sig2.hex(), len(cond_sig2), n=n, print_type="condsig")

"Leaf Node (6, 6) Authentication Path",Value


┌───────────────────────────────────────────────────────────────────────────────┐
│  ■ MTL-Type     ■ Flags               ■ SID                  ■ Randomizer     │
│  ■ Leaf Index   ■ Target Rung Index   ■ Sibling Hash Count   ■ Sibling Hash   │
└───────────────────────────────────────────────────────────────────────────────┘

╭───────────── Condensed Signature For 'the Closest Encloser' ──────────────╮
│                                                                           │
│  0200008F 5CE818A1 DF091DFB 8288CCAD 6FF45099 A48C464F 540CAB05 11498173  │
│  6BB9D07B 3D92E1F4 A6C80B59 2D3F1A64 8E7C9B00 00000000 00000600 00000000  │
│  00000000 00000000 00000700 031E3FB8 0CDF6481 94F2F036 112C841C 299C0032  │
│  23EE1B17 3342B578 AA8C5D3C 56AD692E 7D45185C E82BF093 144A22C7 15        │
│                                                                           │
│                                                                           │
│  Length of Condensed Signature For 'the Closest Encloser': 125 bytes      │
│                                                                           │
╰───────────────────────────────────────────────────────────────────────────╯

In [29]:
# Flag has to be 0
nsec3_3_authpath_bytes = int("0").to_bytes(2, byteorder="big") + \
                         ladder_signed.sid + \
                         nsec3_3_leaf_node.randomizer + \
                         nsec3_3_leaf_node.lindex.to_bytes(8, byteorder="big") + \
                         nsec3_3_leaf_node.rung_lindex + \
                         nsec3_3_leaf_node.rung_rindex + \
                         len(nsec3_3_leaf_node.siblings).to_bytes(2, byteorder="big")

if nsec3_3_leaf_node.siblings:
    for each_sib_hash in nsec3_3_leaf_node.siblings:
        nsec3_3_authpath_bytes += each_sib_hash

nsec3_3_leaf_node.authpath = nsec3_3_authpath_bytes
display(nsec3_3_leaf_node.authpath_print())

cond_sig3 = int(2).to_bytes(1, byteorder="big") + nsec3_3_leaf_node.authpath
print_data("Condensed Signature For 'Wildcard at the Closest Encloser'", 
           cond_sig3.hex(), len(cond_sig3), n=n, print_type="condsig")

"Leaf Node (7, 7) Authentication Path",Value


┌───────────────────────────────────────────────────────────────────────────────┐
│  ■ MTL-Type     ■ Flags               ■ SID                  ■ Randomizer     │
│  ■ Leaf Index   ■ Target Rung Index   ■ Sibling Hash Count   ■ Sibling Hash   │
└───────────────────────────────────────────────────────────────────────────────┘

╭─────────── Condensed Signature For 'Wildcard at the Closest Encloser' ────────────╮
│                                                                                   │
│  0200008F 5CE818A1 DF091DFB 8288CCAD 6FF45099 A48C464F 540CAB05 11498173          │
│  6BB9D0E2 8A5C31FD 490B6781 EA203BC9 57F46200 00000000 00000700 00000000          │
│  00000000 00000000 00000700 03F36D22 7EF01D80 4BAF7747 A16B1AF5 BA9C0032          │
│  23EE1B17 3342B578 AA8C5D3C 56AD692E 7D45185C E82BF093 144A22C7 15                │
│                                                                                   │
│                                                                                   │
│  Length of Condensed Signature For 'Wildcard at the Closest Encloser': 125 bytes  │
│                                                                                   │
╰───────────────────────────────────────────────────────────────────────────────────╯

## 8. Server - Constructing a Response

The server now places **one full signature** into the RRSIG of the SOA record, and **three condensed signatures** into the RRSIGs of three NSEC3 records. Because the client uses SigTag, the server also needs to include an empty SigTag EDNS(0) option in its response to signal its support for SigTag, as follows:
```
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
   |               OPTION-CODE = TBD               |
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
   |             OPTION-LENGTH = 0x0000            |
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+
```
The server then responds to the client with **a 2649-byte full signature**, **three 125-byte condensed signatures**, and **an empty SigTag EDNS(0) option**. 

## 📝 End of Demo

This demo shows how a server performs the deduplication optimization that is enabled by SigTag.